In [ ]:
import pandas as pd
from clase_reportes import ReportClass
rc = ReportClass()
ruta = rc.validar_ruta()
ruta_kits = ruta / 'data' / 'kits.xlsx'
df_kits = pd.read_excel(ruta_kits)


ruta_base =ruta / 'file' / 'BASE VENTAS 2025.xlsx'

# Cargar el archivo de Excel
df_ventas = pd.read_excel(ruta_base)  # Reemplaza con la ruta de tu archivo


# Unir df_ventas con df_kits para explotar los kits
df_explosion = pd.merge(df_ventas, df_kits, left_on="PRODUCTO", right_on="KIT")

# Agregar una columna para indicar el origen (kit o individual)
df_explosion["ORIGEN"] = "KIT"

# Calcular las cantidades de productos
df_explosion["CANTIDAD_PRODUCTO"] = df_explosion["CANTIDAD"]

# Calcular el valor por producto en los kits
df_explosion["VALOR_POR_PRODUCTO"] = df_explosion["TOTAL($)"] / df_explosion.groupby("KIT")["PRODUCTO_x"].transform("count")

# Agrupar y sumar las cantidades de productos de kits
df_resultado_kits = df_explosion.groupby(["PRODUCTO_y", "MES", "ORIGEN"])["CANTIDAD_PRODUCTO"].sum().reset_index()
df_resultado_kits.columns = ["PRODUCTO", "MES","ORIGEN","CANTIDAD_TOTAL"]


# Filtrar productos individuales
df_ventas_individuales = df_ventas[~df_ventas["PRODUCTO"].str.startswith(("[PCNKIT","[TNGKIT","[B8KIT"))].reset_index(drop=True)
df_ventas_individuales["ORIGEN"] = "INDIVIDUAL"

# Seleccionar y renombrar columnas para que coincidan con df_resultado_kits
df_ventas_individuales = df_ventas_individuales[["PRODUCTO", "MES","ORIGEN" ,"CANTIDAD", "TOTAL($)" ]]
df_ventas_individuales.columns = ["PRODUCTO", "MES", "ORIGEN", "CANTIDAD_TOTAL", "INGRESO_TOTAL"]

# Combinar los resultados de kits y productos individuales
df_final = pd.concat([df_resultado_kits, df_ventas_individuales], ignore_index=True)

## ingresos

# Agrupar y sumar las cantidades de productos de kits
df_ingresos_kits = df_explosion.groupby(["PRODUCTO_y", "MES"]).agg({
    "CANTIDAD_PRODUCTO": "sum",
    "VALOR_POR_PRODUCTO": "sum"
}).reset_index()
df_ingresos_kits.columns = ["PRODUCTO", "MES", "CANTIDAD_TOTAL", "INGRESO_TOTAL"]
df_ingresos= pd.concat([df_ingresos_kits, df_ventas_individuales], ignore_index=True)


# Lista con el orden correcto de los meses en español
orden_meses = [
    'ENERO', 'FEBRERO', 'MARZO', 'ABRIL', 'MAYO', 'JUNIO',
    'JULIO', 'AGOSTO', 'SEPTIEMBRE', 'OCTUBRE', 'NOVIEMBRE', 'DICIEMBRE'
]

# Convertir la columna MES a tipo categoría con orden explícito
df_final['MES'] = pd.Categorical(df_final['MES'], categories=orden_meses, ordered=True)

# Ahora ordenar por MES
df_final = df_final.sort_values(by='MES')


# Crear la pivot table
pivot_table_por_mes = df_final.pivot_table(
    index="PRODUCTO",  # Filas: productos
    columns="MES",     # Columnas: meses
    values="CANTIDAD_TOTAL",  # Valores: cantidades
    aggfunc="sum",     # Función de agregación: suma
    fill_value=0,
    observed=True 
              # Rellenar valores faltantes con 0
).reset_index()


# Crear la pivot table
pivot_table_mes_origen = df_final.pivot_table(
    index="PRODUCTO",  # Filas: productos
    columns=["MES", "ORIGEN"],  # Columnas: meses y origen (kit o individual)
    values="CANTIDAD_TOTAL",  # Valores: cantidades
    aggfunc="sum",     # Función de agregación: suma
    fill_value=0,
    observed=True      # Rellenar valores faltantes con 0
).reset_index()


# Crear la pivot table con el nuevo formato
pivot_table_resumida = df_final.pivot_table(
    index=["PRODUCTO", "ORIGEN"],  # Filas: Producto y tipo (Kit o Individual)
    columns="MES",                 # Columnas: Meses
    values="CANTIDAD_TOTAL",        # Valores: Cantidad total
    aggfunc="sum",                  # Sumar cantidades
    fill_value=0,
    observed=True                    # Reemplazar NaN con 0
).reset_index()

## 
# Crear la pivot table para las cantidades
pivot_ingresos_cantidades = df_ingresos.pivot_table(
    index="PRODUCTO",  # Filas: productos
    columns="MES",     # Columnas: meses
    values="CANTIDAD_TOTAL",  # Valores: cantidades
    aggfunc="sum",     # Función de agregación: suma
    fill_value=0       # Rellenar valores faltantes con 0
).reset_index()

# Crear la pivot table para los ingresos
pivot_table_ingresos = df_ingresos.pivot_table(
    index="PRODUCTO",  # Filas: productos
    columns="MES",     # Columnas: meses
    values="INGRESO_TOTAL",  # Valores: ingresos
    aggfunc="sum",     # Función de agregación: suma
    fill_value=0       # Rellenar valores faltantes con 0
).reset_index()


In [ ]:

ruta_file = ruta / 'file' 

# Guardar la pivot table en un archivo de Excel
pivot_table_por_mes.to_excel(ruta_file / 'pivot_table_por_mes.xlsx', index=False) ## Es igual a "pivot_table_cantidades_por_mes.xlsx"
# Guardar la pivot table en un archivo de Excel
pivot_table_mes_origen.to_excel(ruta_file / "pivot_table_por_mes_y_origen.xlsx")
# Guardar la pivot table en un archivo de Excel
pivot_table_resumida.to_excel(ruta_file / "pivot_table_resumida.xlsx", index=False)

pivot_ingresos_cantidades.to_excel(ruta_file /"pivot_table_cantidades_por_mes.xlsx", index=False)

pivot_table_ingresos.to_excel(ruta_file / "pivot_table_ingresos_por_mes.xlsx", index=False)



# ingresos

In [ ]:
import pandas as pd

# Cargar el archivo de Excel con las ventas
df_ventas = pd.read_excel("BASE VENTAS 2025.xlsx")  # Reemplaza con la ruta de tu archivo

# Aplicar los filtros requeridos
df_ventas_filtrado = df_ventas[
    (df_ventas["CATEGORÍA"] == "SHOPIFY") & 
    (df_ventas["CIUDAD_CORREGIDA"] == "CALI")
]

# Mostrar las primeras filas para verificar
print(df_ventas_filtrado.head())

# Definir el orden cronológico de los meses
orden_meses = ['ENERO', 'FEBRERO', 'MARZO', 'ABRIL', 'MAYO', 'JUNIO', 
               'JULIO', 'AGOSTO', 'SEPTIEMBRE', 'OCTUBRE', 'NOVIEMBRE', 'DICIEMBRE']

# Convertir la columna MES a tipo categórico con el orden especificado
df_ventas_filtrado['MES'] = pd.Categorical(
    df_ventas_filtrado['MES'], 
    categories=orden_meses, 
    ordered=True
)




# Datos de los kits y productos (crear el DataFrame de kits)
data = {
    "KIT": [
        "[PCNKIT] KIT LA POCIÓN", "[PCNKIT] KIT LA POCIÓN", "[PCNKIT] KIT LA POCIÓN", "[PCNKIT] KIT LA POCIÓN",
        "[PCNKIT10] KIT NUTRICIÓN INTENSA", "[PCNKIT10] KIT NUTRICIÓN INTENSA", "[PCNKIT10] KIT NUTRICIÓN INTENSA", "[PCNKIT10] KIT NUTRICIÓN INTENSA",
        "[PCNKIT11] KIT DÚO PERFUMES CAPILARES", "[PCNKIT11] KIT DÚO PERFUMES CAPILARES",
        "[PCNKIT12] KIT CRECIMIENTO ACTIVO", "[PCNKIT12] KIT CRECIMIENTO ACTIVO", "[PCNKIT12] KIT CRECIMIENTO ACTIVO", "[PCNKIT12] KIT CRECIMIENTO ACTIVO", "[PCNKIT12] KIT CRECIMIENTO ACTIVO",
        "[PCNKIT15] KIT CONTROL GRASA Y REPARACIÓN", "[PCNKIT15] KIT CONTROL GRASA Y REPARACIÓN", "[PCNKIT15] KIT CONTROL GRASA Y REPARACIÓN", "[PCNKIT15] KIT CONTROL GRASA Y REPARACIÓN",
        "[PCNKIT2] KIT DE REGENERACIÓN PROFUNDA", "[PCNKIT2] KIT DE REGENERACIÓN PROFUNDA", "[PCNKIT2] KIT DE REGENERACIÓN PROFUNDA", "[PCNKIT2] KIT DE REGENERACIÓN PROFUNDA", "[PCNKIT2] KIT DE REGENERACIÓN PROFUNDA",
        "[PCNKIT3] KIT DOBLE PODER REPARADOR", "[PCNKIT3] KIT DOBLE PODER REPARADOR", "[PCNKIT3] KIT DOBLE PODER REPARADOR", "[PCNKIT3] KIT DOBLE PODER REPARADOR", "[PCNKIT3] KIT DOBLE PODER REPARADOR",
        "[PCNKIT4] Kit Efecto Detox", "[PCNKIT4] Kit Efecto Detox", "[PCNKIT4] Kit Efecto Detox", "[PCNKIT4] Kit Efecto Detox", "[PCNKIT4] Kit Efecto Detox",
        "[PCNKIT5] KIT FUERZA Y CRECIMIENTO", "[PCNKIT5] KIT FUERZA Y CRECIMIENTO", "[PCNKIT5] KIT FUERZA Y CRECIMIENTO", "[PCNKIT5] KIT FUERZA Y CRECIMIENTO", "[PCNKIT5] KIT FUERZA Y CRECIMIENTO",
        "[PCNKIT6] KIT DE REPARACIÓN CON DOYPACK", "[PCNKIT6] KIT DE REPARACIÓN CON DOYPACK", "[PCNKIT6] KIT DE REPARACIÓN CON DOYPACK", "[PCNKIT6] KIT DE REPARACIÓN CON DOYPACK",
        "[PCNKIT8] KIT RIZOS BRILLANTES", "[PCNKIT8] KIT RIZOS BRILLANTES", "[PCNKIT8] KIT RIZOS BRILLANTES", "[PCNKIT8] KIT RIZOS BRILLANTES", "[PCNKIT8] KIT RIZOS BRILLANTES",
        "[TNGKIT] KIT TONGOLÉ", "[TNGKIT] KIT TONGOLÉ", "[TNGKIT] KIT TONGOLÉ", "[TNGKIT] KIT TONGOLÉ",
        "[B8KIT] KIT BRILLO INFINITO", "[B8KIT] KIT BRILLO INFINITO",
        "[PCNKIT16] KIT CONTROL GRASA Y CRECIMIENTO", "[PCNKIT16] KIT CONTROL GRASA Y CRECIMIENTO", "[PCNKIT16] KIT CONTROL GRASA Y CRECIMIENTO", "[PCNKIT16] KIT CONTROL GRASA Y CRECIMIENTO", "[PCNKIT16] KIT CONTROL GRASA Y CRECIMIENTO",
        "[PCNKIT13] KIT NUTRICIÓN Y CRECIMIENTO", "[PCNKIT13] KIT NUTRICIÓN Y CRECIMIENTO", "[PCNKIT13] KIT NUTRICIÓN Y CRECIMIENTO", "[PCNKIT13] KIT NUTRICIÓN Y CRECIMIENTO",
        "[PCNKIT17] KIT RIZOS LARGO Y ABUNDANTES", "[PCNKIT17] KIT RIZOS LARGO Y ABUNDANTES", "[PCNKIT17] KIT RIZOS LARGO Y ABUNDANTES", "[PCNKIT17] KIT RIZOS LARGO Y ABUNDANTES", "[PCNKIT17] KIT RIZOS LARGO Y ABUNDANTES",
        "[PCNKIT14] KIT RIZOS LARGOS E HIDRATADOS", "[PCNKIT14] KIT RIZOS LARGOS E HIDRATADOS", "[PCNKIT14] KIT RIZOS LARGOS E HIDRATADOS", "[PCNKIT14] KIT RIZOS LARGOS E HIDRATADOS",
        "[B8KIT] KIT BRILLO INFINITO", "[B8KIT] KIT BRILLO INFINITO"
    ],
    "PRODUCTO": [
        "[PCN02] SHAMPOO LA POCION", "[PCN01] TRATAMIENTO LA POCION", "[PCN03] DUAL CREMA PARA PEINAR", "[PCN04] MASCARILLA ANCESTRAL PARA EL CABELLO",
        "[PCN02] SHAMPOO LA POCION", "[PCN01] TRATAMIENTO LA POCION", "[PCN03] DUAL CREMA PARA PEINAR", "[PCN21] MASCARILLA BITE ME",
        "[PCN10] PERFUME CAPILAR PRAIA", "[PCN20] PERFUME CAPILAR BOREAL",
        "[PCN19] DUTONIC (TONICO CAPILAR)", "[PCN01] TRATAMIENTO LA POCION", "[PCN03] DUAL CREMA PARA PEINAR", "[PCN04] MASCARILLA ANCESTRAL PARA EL CABELLO", "[PCN26] SHAMPOO CRECIMIENTO Y CAIDA",
        "[PCN25] SHAMPOO CONTROL GRASA POCION", "[PCN01] TRATAMIENTO LA POCION", "[PCN03] DUAL CREMA PARA PEINAR", "[PCN04] MASCARILLA ANCESTRAL PARA EL CABELLO",
        "[PCN02] SHAMPOO LA POCION", "[PCN01] TRATAMIENTO LA POCION", "[PCN03] DUAL CREMA PARA PEINAR", "[PCN04] MASCARILLA ANCESTRAL PARA EL CABELLO", "[PCN07] OLEO BRILLO INFINITO",
        "[PCN02] SHAMPOO LA POCION", "[PCN01] TRATAMIENTO LA POCION", "[PCN03] DUAL CREMA PARA PEINAR", "[PCN04] MASCARILLA ANCESTRAL PARA EL CABELLO", "[PCN10] PERFUME CAPILAR PRAIA",
        "[PCN02] SHAMPOO LA POCION", "[PCN01] TRATAMIENTO LA POCION", "[PCN03] DUAL CREMA PARA PEINAR", "[PCN04] MASCARILLA ANCESTRAL PARA EL CABELLO", "[PCN20] PERFUME CAPILAR BOREAL",
        "[PCN02] SHAMPOO LA POCION", "[PCN01] TRATAMIENTO LA POCION", "[PCN03] DUAL CREMA PARA PEINAR", "[PCN04] MASCARILLA ANCESTRAL PARA EL CABELLO", "[PCN19] DUTONIC (TONICO CAPILAR)",
        "[PCN11] DOYPACK SHAMPOO", "[PCN01] TRATAMIENTO LA POCION", "[PCN03] DUAL CREMA PARA PEINAR", "[PCN04] MASCARILLA ANCESTRAL PARA EL CABELLO",
        "[TNGL01] SHAMPOO TONGOLÉ", "[TNGL02] TRATAMIENTO TONGOLÉ", "[TNGL03] LEAVE ON TONGOLÉ", "[TNGL04] MASCARILLA TONGOLÉ", "[PCN07] OLEO BRILLO INFINITO",
        "[TNGL01] SHAMPOO TONGOLÉ", "[TNGL02] TRATAMIENTO TONGOLÉ", "[TNGL03] LEAVE ON TONGOLÉ", "[TNGL04] MASCARILLA TONGOLÉ",
        "[PCN07] OLEO BRILLO INFINITO", "[PCN09] TERMOPROTECTOR BRILLO INFINITO",
        "[PCN25] SHAMPOO CONTROL GRASA POCION", "[PCN01] TRATAMIENTO LA POCION", "[PCN03] DUAL CREMA PARA PEINAR", "[PCN04] MASCARILLA ANCESTRAL PARA EL CABELLO", "[PCN19] DUTONIC (TONICO CAPILAR)",
        "[PCN26] SHAMPOO CRECIMIENTO Y CAIDA", "[PCN01] TRATAMIENTO LA POCION", "[PCN21] MASCARILLA BITE ME", "[PCN19] DUTONIC (TONICO CAPILAR)",
        "[TNGL01] SHAMPOO TONGOLÉ", "[TNGL02] TRATAMIENTO TONGOLÉ", "[TNGL03] LEAVE ON TONGOLÉ", "[TNGL04] MASCARILLA TONGOLÉ", "[PCN19] DUTONIC (TONICO CAPILAR)",
        "[PCN26] SHAMPOO CRECIMIENTO Y CAIDA", "[TNGL02] TRATAMIENTO TONGOLÉ", "[TNGL03] LEAVE ON TONGOLÉ", "[TNGL04] MASCARILLA TONGOLÉ",
        "[PCN07] OLEO BRILLO INFINITO", "[PCN09] TERMOPROTECTOR BRILLO INFINITO"
    ]
}

# Unir df_ventas con df_kits para explotar los kits
df_explosion = pd.merge(df_ventas_filtrado, df_kits, left_on="PRODUCTO", right_on="KIT")

# Calcular las cantidades de productos
df_explosion["CANTIDAD_PRODUCTO"] = df_explosion["CANTIDAD"]

# Calcular el valor por producto en los kits
df_explosion["VALOR_POR_PRODUCTO"] = df_explosion["TOTAL($)"] / df_explosion.groupby("KIT")["PRODUCTO_x"].transform("count")

# Agrupar y sumar las cantidades de productos de kits
df_resultado_kits = df_explosion.groupby(["PRODUCTO_y", "MES"]).agg({
    "CANTIDAD_PRODUCTO": "sum",
    "VALOR_POR_PRODUCTO": "sum"
}).reset_index()
df_resultado_kits.columns = ["PRODUCTO", "MES", "CANTIDAD_TOTAL", "INGRESO_TOTAL"]

# Filtrar productos individuales
df_ventas_individuales = df_ventas_filtrado[
    ~df_ventas_filtrado["PRODUCTO"].str.startswith("[PCNKIT") &
    ~df_ventas_filtrado["PRODUCTO"].str.startswith("[TNGKIT") &
    ~df_ventas_filtrado["PRODUCTO"].str.startswith("[B8KIT")
]

# Seleccionar y renombrar columnas para que coincidan con df_resultado_kits
df_ventas_individuales = df_ventas_individuales[["PRODUCTO", "MES", "CANTIDAD", "TOTAL($)"]]
df_ventas_individuales.columns = ["PRODUCTO", "MES", "CANTIDAD_TOTAL", "INGRESO_TOTAL"]

# Combinar los resultados de kits y productos individuales
df_final = pd.concat([df_resultado_kits, df_ventas_individuales], ignore_index=True)

# Asegurarse de que la columna MES en df_final también tenga el orden correcto
df_final['MES'] = pd.Categorical(
    df_final['MES'], 
    categories=orden_meses, 
    ordered=True
)

# Crear la pivot table para las cantidades
pivot_table_cantidades = df_final.pivot_table(
    index="PRODUCTO",
    columns="MES",
    values="CANTIDAD_TOTAL",
    aggfunc="sum",
    fill_value=0
)

# Reordenar las columnas según el orden de meses
pivot_table_cantidades = pivot_table_cantidades[orden_meses]

# Crear la pivot table para los ingresos
pivot_table_ingresos = df_final.pivot_table(
    index="PRODUCTO",
    columns="MES",
    values="INGRESO_TOTAL",
    aggfunc="sum",
    fill_value=0
)

# Reordenar las columnas según el orden de meses
pivot_table_ingresos = pivot_table_ingresos[orden_meses]

# Mostrar las pivot tables
print("Pivot Table de Cantidades:")
print(pivot_table_cantidades)

print("\nPivot Table de Ingresos:")
print(pivot_table_ingresos)

# Guardar las pivot tables en archivos de Excel (opcional)
pivot_table_cantidades.to_excel("pivot_table_cantidades_por_mes_DROPPI.xlsx")
pivot_table_ingresos.to_excel("pivot_table_ingresos_por_mes_DROPPI.xlsx")